# AWS Module 5 – Networking
**Status:** ⬜ Not started | **Course:** AWS Cloud Practitioner Essentials

**Lessons:** Intro to Networking → Organizing AWS Cloud Resources → More Ways to Connect (reading) → Subnets, Security Groups & NACLs → Amazon VPC Demo → Global Networking → Global Architectures

---

## MODULE 5 INTRODUCTION

### Introduction to Networking

- **Networking** = how your AWS resources connect to each other, to your on-premises network, and to the internet
- Two jobs: **let the right traffic in** and **keep the wrong traffic out**
- The foundation of AWS networking is the **Amazon VPC**

---

## NETWORKING

### Organizing AWS Cloud Resources
**Amazon VPC (Virtual Private Cloud)** = your own **logically isolated** section of the AWS Cloud where you launch resources in a virtual network you define.

- You choose the **IP address range** (CIDR block, e.g., `10.0.0.0/16`)
- A VPC lives in **one Region** and can span all of that Region's AZs

| Component | What it does | Analogy (office building) |
|---|---|---|
| **Subnet** | A section of the VPC (lives in **one AZ**) that groups resources | Floors of the building |
| **Public subnet** | Has a route to the internet (e.g., web servers, load balancers) | Lobby open to visitors |
| **Private subnet** | No direct route from the internet (e.g., databases, back-end apps) | Locked employee-only floors |
| **Internet gateway (IGW)** | Connects the VPC to the **public internet** | Front door to the street |
| **Virtual private gateway (VGW)** | AWS-side endpoint for an **encrypted VPN** from your private network | Badge-only side entrance for approved visitors |
| **NAT gateway** | Lets private-subnet resources reach **out** to the internet (e.g., for updates) without being reachable **from** it | Mail room: staff can send out, strangers can't walk in |
| **Route table** | Rules that decide where network traffic goes | Building directory and signs |

### More Ways to Connect to the AWS Cloud (reading)

| Option | How it works | Best for |
|---|---|---|
| **AWS Site-to-Site VPN** | Encrypted tunnel between your **whole network** and your VPC, over the public internet | Quick, low-cost hybrid connectivity |
| **AWS Client VPN** | Encrypted connection for **individual users** (e.g., remote employees) | Remote workforce access |
| **AWS Direct Connect** | **Dedicated private line** from your data center to AWS; doesn't use the public internet | Consistent performance, high bandwidth, lower data transfer costs, compliance |
| **AWS PrivateLink** | Private connection to AWS services or other VPCs without traffic touching the internet | Securely reaching services/SaaS privately |
| **AWS Transit Gateway** | Central **hub** connecting many VPCs and on-prem networks | Simplifying large, complex networks |

> 💡 **VPN vs Direct Connect:** VPN = encrypted but rides the shared internet (fast to set up). Direct Connect = private dedicated line (consistent, higher bandwidth, takes longer to set up).

### Subnets, Security Groups, and Network Access Control Lists
Data travels across a network in small units called **packets**. Two layers of virtual firewalls inspect them:

| | **Network ACL (NACL)** | **Security Group** |
|---|---|---|
| Protects | The **subnet** | The **instance** (e.g., an EC2 instance) |
| State | **Stateless**: checks every packet in **both** directions, remembers nothing | **Stateful**: remembers allowed requests and lets the response back in automatically |
| Rule types | **Allow and deny** | **Allow only** |
| Default | Default NACL **allows all** inbound and outbound (a new custom NACL denies all until you add rules) | **Denies all inbound**, **allows all outbound** |
| Rule order | Evaluated in **number order**, first match wins | All rules evaluated together |

**Analogy:**

- **NACL = passport control at the border.** Checks you when you **enter and again when you leave** the country (subnet)
- **Security group = the doorman at your building.** Remembers you on the way in, so you walk right back out

> 💡 Defense in depth: use **both**. NACLs as a coarse subnet-wide filter, security groups as fine-grained per-instance rules.

### Amazon VPC Demo
Typical steps in the console:

1. **Create a VPC** and choose its IP range (the "VPC and more" wizard can build everything below in one go)
2. **Create subnets** in different AZs: public and private
3. **Create an internet gateway** and **attach** it to the VPC
4. **Update the public subnet's route table:** send internet traffic (`0.0.0.0/0`) to the internet gateway
5. **Create security groups** (e.g., allow HTTPS from the internet to web servers only)
6. **Launch resources** (e.g., EC2) into the right subnets

### Global Networking
**DNS (Domain Name System)** = the internet's phone book: translates names (`example.com`) into IP addresses.

| Service | What it does |
|---|---|
| **Amazon Route 53** | AWS's highly available **DNS** service. Registers domains, routes users to apps, and runs **health checks** |
| **Amazon CloudFront** | **CDN** that caches content at **edge locations** close to users |
| **AWS Global Accelerator** | Routes user traffic over the **AWS private global network** (not the public internet) for better performance and automatic failover |

**Route 53 routing policies (know the names):**

| Policy | Routes users… |
|---|---|
| **Simple** | To a single resource |
| **Latency-based** | To the Region with the lowest latency for that user |
| **Geolocation** | Based on the user's location (e.g., EU users → EU site) |
| **Weighted** | By percentages (e.g., 90% old version / 10% new version) |
| **Failover** | To a backup when the primary fails a health check |

**How a request flows:** User → **Route 53** (DNS lookup) → nearest **CloudFront edge location** → (on cache miss) **Application Load Balancer** → **EC2** instances → response back to user

### Global Architectures
Two reference patterns:

**1. Direct Connect failover (hybrid, fault tolerant)**

- Customer data center → router/firewall → **two** Direct Connect connections (or Direct Connect + VPN backup) → **virtual private gateway** → VPC
- If one connection fails, traffic keeps flowing over the other
- Benefit: **high bandwidth + fault tolerance** for critical apps and large transfers

**2. Multi-Region content delivery (global, low latency)**

- User → custom domain → **Route 53** (routes to the closest Region/edge) → **CloudFront** edge cache → **origin servers spread across multiple AZs**
- Benefit: **low latency worldwide + high availability**

---

## MODULE 5 CONCLUSION

### Module Summary

- **VPC** = your private, isolated network in AWS; split into **public/private subnets** across AZs
- **IGW** = internet access | **VGW** = VPN endpoint | **NAT gateway** = outbound-only internet for private subnets
- Connect on-prem with **VPN** (over the internet) or **Direct Connect** (dedicated private line)
- **NACL** = subnet, stateless | **Security group** = instance, stateful
- **Route 53** = DNS | **CloudFront** = CDN | **Global Accelerator** = AWS backbone routing

---

## 🎯 Databricks SA Connection

- **One of the most important modules for regulated deals.** Network and security teams will ask exactly how Databricks fits into their network, often before anything else
- **Customer-managed VPC:** Databricks classic compute can be deployed into the **customer's own VPC and subnets**, under their security groups and route tables
- **Secure cluster connectivity:** cluster nodes have **no public IPs and no open inbound ports**; they connect out to the Databricks control plane
- **PrivateLink:** keeps traffic between users, the Databricks control plane, and compute **off the public internet**. A very common requirement in banking, healthcare, and government
- **NAT gateway costs:** clusters pulling large amounts of data through a NAT gateway can get expensive. An **S3 gateway VPC endpoint** keeps S3 traffic on the AWS network and avoids those charges
- **IP access lists** restrict which corporate networks can reach the workspace
- **Discovery questions:** "Do you require private connectivity / no public IPs?" and "Do you bring your own VPC?" The answers shape the whole deployment architecture

---

## 📝 Exam / Interview Quick Hits

- **VPC** = logically isolated network; **subnets** live in one AZ
- **Internet gateway** = public internet | **Virtual private gateway** = VPN | **NAT gateway** = private subnets reach out only
- **Site-to-Site VPN** = encrypted over the internet | **Direct Connect** = dedicated private line | **Client VPN** = individual users
- **NACL** = subnet, **stateless**, allow + deny | **Security group** = instance, **stateful**, allow only
- **Route 53** = DNS + routing policies + health checks | **CloudFront** = CDN | **Global Accelerator** = AWS global network